In [ ]:
import collections
import random
import decimal
import math
import sys

def Query(item,id,FP1,FP2,FP3):
    if len(Level[3][id]) > 0:
        for index,value in enumerate(Level[3][id]):
            if value != 0:
                if value[0] == FP3:
                    return [index,3]
    if len(Level[2][id]) > 0:
        for index, value in enumerate(Level[2][id]):
            if value != 0:
                if value[0] == FP2:
                    return [index, 2]
    if len(Level[1][id]) > 0:
        for index,value in enumerate(Level[1][id]):
            if value != 0:
                if value[0] == FP1:
                    return [index,1]
    return 0

def Levelup(k):
    if k == 2:
        for index,item in enumerate(Level[3][bucket_index1]):
            if item == 0:
                Level[3][bucket_index1][index] = [FP_level3, 2 ** counter_bit_level2]
                return True
        if len(Level[1][bucket_index1]) >= 3:
            Clear(Level[1][bucket_index1], 3)
            Level[3][bucket_index1].append([FP_level3, 2 ** counter_bit_level2])
            return True
        elif len(Level[2][bucket_index1]) >= 3:
            Clear(Level[2][bucket_index1], 3)
            Level[3][bucket_index1].append([FP_level3, 2 ** counter_bit_level2])
            Level[3][bucket_index1].append(0)
            return True
        return False

    elif k == 1:
        for index,item in enumerate(Level[2][bucket_index1]):
            if item == 0:
                Level[2][bucket_index1][index] = [FP_level2, 2 ** counter_bit_level1]
                return True
        if len(Level[1][bucket_index1]) >= 3:
            Clear(Level[1][bucket_index1], 3)
            Level[2][bucket_index1].append([FP_level2, 2 ** counter_bit_level1])
            Level[2][bucket_index1].append(0)
            return True
        return False

def Clear(lst,clear_num):
    for times in range(clear_num):
        Delete_Min(lst)

def Delete_Min(lst):
    min_num = sys.maxsize
    min_pos = -1
    for i in range(len(lst)):
        if lst[i] != 0:
            if lst[i][1] < min_num:
                min_num = lst[i][1]
                min_pos = i
        else:
            lst.pop(i)
            return lst

    lst.pop(min_pos)
    return lst

def Exponential_Decay(lst,k):
    if len(lst) == 0:
        return False
    min_num = sys.maxsize
    min_pos = -1
    for i in range(len(lst)):
        if lst[i] != 0:
            if isinstance(lst[i][1],int):
                if lst[i][1] < min_num:
                    min_num = lst[i][1]
                    min_pos = i
    if min_num == sys.maxsize:
        gailv_minus = 0
    else:
        gailv_minus = 1 / b ** math.log2(min_num)

    if random.random() < gailv_minus:
        if isinstance(lst[min_pos][1], int):
            lst[min_pos][1] -= 1
            if lst[min_pos][1] < 2**k:
                if k == counter_bit_level1:
                    lst[min_pos][0] = FP_level2
                elif k == counter_bit_level2:
                    lst[min_pos][0] = FP_level3
                lst[min_pos][1] = 2**k
                return True
    return False

In [ ]:
# load the dataset
F = []

with open("dataset.txt", "r") as file:

    for line in file:
        
        f = line # item ID
        
        F.append(f)

print('The numbers of items:{}'.format(len(F)))
l = collections.Counter(F)
print('The numbers of unique items:{}'.format(len(l)))

In [ ]:
import torch
import torch.nn as nn
import numpy as np

class UnsupervisedClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            
            nn.Linear(128, 512),
            nn.LayerNorm(512),
            nn.ReLU(),
            
            nn.Linear(512, num_classes)
        )

    def forward(self, x, tau=1.0):
        logits = self.net(x)
        return logits

def ip_to_tensor(ip_list):
    parts = [[int(x) for x in ip.split(".")] for ip in ip_list]
    return torch.tensor(parts).float() / 255.0

# load tuned Dispacher

model = UnsupervisedClassifier(input_dim=4, hidden_dim=512, num_classes=K).to(device)

model.load_state_dict(torch.load("fine_tuned_Dispatcher.pt", map_location=device))

model.eval()


In [ ]:
for memory in [20,40,60,80,100]:
    
    T = 750
    topK = 4096
    d = 8
    b = 1.08
    memorysize = memory*128/(128+9)
    
    counter_bit_level1 = 8
    fp_bit_level1 = 8

    counter_bit_level2 = 12
    fp_bit_level2 = 12

    counter_bit_level3 = 20
    fp_bit_level3 = 16
    
    W = (counter_bit_level1+fp_bit_level1)*d

    cell_num = int(W / counter_bit_level1 / 2)
    BUCKET_NUM = int(memorysize*1024*8/W)
    print('Memory Size:{}KB'.format(memorysize))
    print('d:{}'.format(d))
    print('T:{}'.format(T))
    
    structure = [] # obtain from Constructor, omit here
    
    Level1 = []
    Level2 = []
    Level3 = []
    for _ in range(BUCKET_NUM):
        Level1.append([0 for j in range(structure[_][0])])
        Level2.append([0 for j in range(structure[_][1])])
        Level3.append([0 for j in range(structure[_][2])])
    Level = [0,Level1,Level2,Level3]
    
    SEED = [2132, 315, 1651, 3165, 4651]
    
    map_dict = {}
    
    for item in F:
        
        ips = ip_to_tensor([item])[0]
        x = torch.cat([ips], dim=0)
        
        base_out = model(x.unsqueeze(0))
        
        bucket_index1 = torch.argmax(base_out, dim=1)[0].item()
        map_dict[item] = bucket_index1
        
        FP_level1 = mmh3.hash(item,SEED[1]) % (2 ** fp_bit_level1)
        FP_level2 = mmh3.hash(item,SEED[2]) % (2 ** fp_bit_level2)
        FP_level3 = mmh3.hash(item,SEED[3]) % (2 ** fp_bit_level3)
        query_result = Query(item, bucket_index1, FP_level1, FP_level2, FP_level3)
        if query_result != 0:
            query_index = query_result[0]
            query_level = query_result[1]
            if query_level == 3:
                Level[3][bucket_index1][query_index][1] += 1
            elif query_level == 2:
                if Level[2][bucket_index1][query_index][1] < 2**counter_bit_level2-1:
                    Level[2][bucket_index1][query_index][1] += 1
                else:
                    # swap first
                    swap_flag = False
                    if len(Level[3][bucket_index1]) > 0:
                        for index,value in enumerate(Level[3][bucket_index1]):
                            if not swap_flag:
                                if value != 0:
                                    if value[1] < 2**counter_bit_level2 + 1:
                                        tempF = value[0] % (2 ** fp_bit_level2)
                                        tempV = value[1]
                                        Level[3][bucket_index1][index][0] = FP_level3
                                        Level[3][bucket_index1][index][1] = Level[2][bucket_index1][query_index][1] + 1
                                        Level[2][bucket_index1][query_index][0] = tempF
                                        Level[2][bucket_index1][query_index][1] = tempV
                                        swap_flag = True
                                elif value == 0:
                                    Level[3][bucket_index1][index] = [FP_level3, 2**counter_bit_level2]
                                    Level[2][bucket_index1][query_index] = 0
                                    swap_flag = True
                    # then levelup
                    if not swap_flag:
                        query_FP = Level[2][bucket_index1][query_index][0]
                        if Levelup(2):
                            for index, item in enumerate(Level[2][bucket_index1]):
                                if item != 0:
                                    if item[0] == query_FP:
                                        Level[2][bucket_index1][index] = 0
                        else:
                            if Exponential_Decay(Level[3][bucket_index1], counter_bit_level2):
                                Level[2][bucket_index1][query_index] = 0
            elif query_level == 1:
                if Level[1][bucket_index1][query_index][1] < 2 ** counter_bit_level1 - 1:
                    Level[1][bucket_index1][query_index][1] += 1
                else:
                    # swap first
                    swap_flag = False
                    if len(Level[2][bucket_index1]) > 0:
                        for index, value in enumerate(Level[2][bucket_index1]):
                            if not swap_flag:
                                if value != 0:
                                    if value[1] < 2**counter_bit_level1 + 1:
                                        tempF = value[0] % (2 ** fp_bit_level1)
                                        tempV = value[1]
                                        Level[2][bucket_index1][index][0] = FP_level2
                                        Level[2][bucket_index1][index][1] = Level[1][bucket_index1][query_index][1] + 1
                                        Level[1][bucket_index1][query_index][0] = tempF
                                        Level[1][bucket_index1][query_index][1] = tempV
                                        swap_flag = True
                                elif value == 0:
                                    Level[2][bucket_index1][index] = [FP_level2, 2**counter_bit_level1]
                                    Level[1][bucket_index1][query_index] = 0
                                    swap_flag = True
                    if not swap_flag:
                        if len(Level[3][bucket_index1]) > 0:
                            for index,value in enumerate(Level[3][bucket_index1]):
                                if not swap_flag:
                                    if value != 0:
                                        if value[1] < 2**counter_bit_level1 + 1:
                                            tempF = value[0] % (2 ** fp_bit_level1)
                                            tempV = value[1]
                                            Level[3][bucket_index1][index][0] = FP_level3
                                            Level[3][bucket_index1][index][1] = Level[1][bucket_index1][query_index][1] + 1
                                            Level[1][bucket_index1][query_index][0] = tempF
                                            Level[1][bucket_index1][query_index][1] = tempV
                                            swap_flag = True
                                    elif value == 0:
                                        Level[3][bucket_index1][index] = [FP_level3, 2**counter_bit_level1]
                                        Level[1][bucket_index1][query_index] = 0
                                        swap_flag = True
                    # then levelup
                    if not swap_flag:
                        query_FP = Level[1][bucket_index1][query_index][0]
                        if Levelup(1):
                            for index,item in enumerate(Level[1][bucket_index1]):
                                if item != 0:
                                    if item[0] == query_FP:
                                        Level[1][bucket_index1][index] = 0
                        else:
                            if Exponential_Decay(Level[2][bucket_index1], counter_bit_level1):
                                Level[1][bucket_index1][query_index] = 0
        else:
            noEmpty = True
            for index, item in enumerate(Level[1][bucket_index1]):
                if item == 0:
                    Level[1][bucket_index1][index] = [FP_level1, 1]
                    noEmpty = False
                    break
            if noEmpty:
                for index, item in enumerate(Level[2][bucket_index1]):
                    if item == 0:
                        Level[2][bucket_index1][index] = [FP_level2, 1]
                        noEmpty = False
                        break
            if noEmpty:
                for index, item in enumerate(Level[3][bucket_index1]):
                    if item == 0:
                        Level[3][bucket_index1][index] = [FP_level3, 1]
                        noEmpty = False
                        break
            if noEmpty:
                min_num_level1 = sys.maxsize
                min_pos_level1 = -1
                min_level = -1
                for i in range(len(Level[1][bucket_index1])):
                    if Level[1][bucket_index1] != 0:
                        if Level[1][bucket_index1][i][1] < min_num_level1:
                            min_num_level1 = Level[1][bucket_index1][i][1]
                            min_pos_level1 = i
                            min_level = 1
                for i in range(len(Level[2][bucket_index1])):
                    if Level[2][bucket_index1] != 0:
                        if Level[2][bucket_index1][i][1] < min_num_level1:
                            min_num_level1 = Level[2][bucket_index1][i][1]
                            min_pos_level1 = i
                            min_level = 2
                for i in range(len(Level[3][bucket_index1])):
                    if Level[3][bucket_index1] != 0:
                        if Level[3][bucket_index1][i][1] < min_num_level1:
                            min_num_level1 = Level[3][bucket_index1][i][1]
                            min_pos_level1 = i
                            min_level = 3

                if min_num_level1 > 9000:
                    gailv_minus = 0
                else:
                    gailv_minus = 1/(b**(math.log2(min_num_level1)))

                    # replace
                    if random.random() < gailv_minus:
                        Level[min_level][bucket_index1][min_pos_level1][1] -= 1

                        if Level[min_level][bucket_index1][min_pos_level1][1] == 0:
                            Level[min_level][bucket_index1][min_pos_level1][1] = 1
                            if min_level == 1:
                                Level[min_level][bucket_index1][min_pos_level1][0] = FP_level1
                            elif min_level == 2:
                                Level[min_level][bucket_index1][min_pos_level1][0] = FP_level2
                            elif min_level == 3:
                                Level[min_level][bucket_index1][min_pos_level1][0] = FP_level3
    
    
    # detection
    l1 = l.most_common(topK)
    count = 0
    #estimated
    result_overT = {}
    result_all = {}
    #real
    real = {}

    for i in l.items():
        if i[1] > T:
            real.update({i[0]:i[1]})

        ID1 = map_dict[i[0]]
        FP_l1 = mmh3.hash(i[0], SEED[1]) % (2 ** fp_bit_level1)
        FP_l2 = mmh3.hash(i[0], SEED[2]) % (2 ** fp_bit_level2)
        FP_l3 = mmh3.hash(i[0], SEED[3]) % (2 ** fp_bit_level3)
        query_result1 = Query(i[0], ID1,FP_l1,FP_l2,FP_l3)
        
        value1 = 0
        if query_result1 != 0:
            query_index = query_result1[0]
            query_level = query_result1[1]
            if query_level == 3:
                value1 += Level[3][ID1][query_index][1]
            elif query_level == 2:
                value1 += Level[2][ID1][query_index][1]
            else:
                value1 += Level[1][ID1][query_index][1]

        if value1 > T:
            result_overT.update({i[0]: value1})
        result_all.update({i[0]: value1})

    aae_overT = 0
    aae_all = 0
    
    for key in result_overT.keys():
        if key in real:
            count += 1

        aae_overT += abs(result_overT[key] - l[key])

        aae_all += abs(result_all[key] - l[key])

    dic = sorted(result_all.items(), key=lambda x: x[1], reverse=True)[:topK]

    count_topk = 0
    dic_l1 = dict(l1)
    aae_topk = 0
    for item in dic:
        aae_topk += abs(l[item[0]] - item[1])
        if item[0] in dic_l1:
            count_topk += 1
            del dic_l1[item[0]]
    
    print("####################### Threshold-t ############################")
    print('OverT aae:{}'.format(aae_overT/len(result_overT)))
    pr = count / len(result_overT)
    rr = count / len(real)
    print('Threshold={},PR:{}'.format(T, pr))
    print('Threshold={},RR:{}'.format(T, rr))
    print('Threshold={},F_beta:{}'.format(T, ((1+4)*pr*rr)/(4*pr+rr) ))
    
    print("####################### Top-k ############################")
    print('Top-theta aae:{}'.format(aae_topk/topK))
    pr = count_topk / topK
    rr = count_topk / topK
    print('Find top-{},PR:{}'.format(topK, pr))
    print('Find top-{},RR:{}'.format(topK, rr))
    print('Find top-{},F_beta:{}'.format(topK, ((1+4)*pr*rr)/(4*pr+rr) ))
    print("############################################################")
    print()
    